# News Data Exploration + Anecdote Detection

This notebook:
- Loads and explores files in `news_data/`
- Computes basic article length statistics by topic
- Initializes a SambaNova client
- Prepares (but does not run automatically) a per-topic anecdote detection loop that saves JSON outputs


In [1]:
import json
import numpy as np
import re
from pathlib import Path

import pandas as pd
from openai import OpenAI

DATA_DIR = Path("news_data")
OUTPUT_ROOT = Path("anecdote_outputs")
OUTPUT_ROOT.mkdir(exist_ok=True)

ANECDOTE_RESULTS_DIR = OUTPUT_ROOT / "anecdote_detection_results"
ANECDOTE_ONLY_STANCE_DIR = OUTPUT_ROOT / "anecdote_only_stance"
ANECDOTE_SENTENCES_DIR = OUTPUT_ROOT / "anecdote_sentences_only"

for _dir in [ANECDOTE_RESULTS_DIR, ANECDOTE_ONLY_STANCE_DIR, ANECDOTE_SENTENCES_DIR]:
    _dir.mkdir(parents=True, exist_ok=True)

from tqdm.auto import tqdm

In [2]:
def topic_from_filename(path: Path) -> str:
    name = path.stem  # e.g., historical_climate_articles_full
    if name.startswith("historical_") and name.endswith("_articles_full"):
        return name[len("historical_") : -len("_articles_full")]
    return name


def load_articles_file(path: Path) -> list[dict]:
    """Load a JSON articles file, applying light repair for known malformed patterns."""
    raw = path.read_text(encoding="utf-8")
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        # Repair known bad token like: "uri":"244308574",""
        repaired = raw.replace(',""\n        "lang"', ',\n        "lang"')
        data = json.loads(repaired)

    if not isinstance(data, list):
        raise ValueError(f"Expected list of records in {path}, got {type(data).__name__}")
    return data


def load_all_articles(data_dir: Path) -> pd.DataFrame:
    frames = []
    for file_path in sorted(data_dir.glob("*.json")):
        topic = topic_from_filename(file_path)
        rows = load_articles_file(file_path)
        df = pd.DataFrame(rows)
        df["topic"] = topic
        df["source_file"] = file_path.name
        frames.append(df)

    all_df = pd.concat(frames, ignore_index=True)
    all_df["body"] = all_df.get("body", "").fillna("").astype(str)
    return all_df

In [3]:
articles_df = load_all_articles(DATA_DIR)

print(f"Total articles: {len(articles_df):,}")
print(f"Topics: {sorted(articles_df['topic'].dropna().unique().tolist())}")
print("")

summary = (
    articles_df.groupby("topic", as_index=False)
    .agg(article_count=("body", "size"))
    .sort_values("article_count", ascending=False)
)
summary

Total articles: 9,049
Topics: ['2_historical_DP_articles_full', '2_historical_climate_articles_full', '2_historical_gun_articles_full', 'DP', 'RR', 'climate', 'gun', 'immigration']



,topic,article_count
7,immigration,1684
1,2_historical_climate_articles_full,1559
5,climate,1406
2,2_historical_gun_articles_full,1114
4,RR,947
6,gun,907
0,2_historical_DP_articles_full,733
3,DP,699


In [4]:
# Basic article length stats by topic
articles_df["body_char_len"] = articles_df["body"].str.len()
articles_df["body_word_len"] = articles_df["body"].str.split().str.len()

length_stats = (
    articles_df.groupby("topic", as_index=False)
    .agg(
        n_articles=("body", "size"),
        char_mean=("body_char_len", "mean"),
        char_median=("body_char_len", "median"),
        char_min=("body_char_len", "min"),
        char_max=("body_char_len", "max"),
        word_mean=("body_word_len", "mean"),
        word_median=("body_word_len", "median"),
        word_min=("body_word_len", "min"),
        word_max=("body_word_len", "max"),
    )
    .sort_values("topic")
)

# Pretty print
for _, row in length_stats.iterrows():
    print(f"Topic: {row['topic']}")
    print(f"  Articles: {int(row['n_articles'])}")
    print(
        f"  Characters -> mean: {row['char_mean']:.1f}, median: {row['char_median']:.1f}, "
        f"min: {int(row['char_min'])}, max: {int(row['char_max'])}"
    )
    print(
        f"  Words      -> mean: {row['word_mean']:.1f}, median: {row['word_median']:.1f}, "
        f"min: {int(row['word_min'])}, max: {int(row['word_max'])}"
    )
    print()

length_stats

Topic: 2_historical_DP_articles_full
  Articles: 733
  Characters -> mean: 5850.0, median: 4355.0, min: 724, max: 122500
  Words      -> mean: 964.1, median: 716.0, min: 112, max: 20142

Topic: 2_historical_climate_articles_full
  Articles: 1559
  Characters -> mean: 6455.5, median: 4602.0, min: 356, max: 124914
  Words      -> mean: 1050.8, median: 741.0, min: 60, max: 22572

Topic: 2_historical_gun_articles_full
  Articles: 1114
  Characters -> mean: 8560.5, median: 4988.5, min: 331, max: 199499
  Words      -> mean: 1445.4, median: 830.5, min: 54, max: 35758

Topic: DP
  Articles: 699
  Characters -> mean: 5161.8, median: 4395.0, min: 491, max: 51806
  Words      -> mean: 854.4, median: 706.0, min: 85, max: 8219

Topic: RR
  Articles: 947
  Characters -> mean: 6312.8, median: 5443.0, min: 200, max: 72982
  Words      -> mean: 1038.3, median: 896.0, min: 33, max: 12439

Topic: climate
  Articles: 1406
  Characters -> mean: 6669.3, median: 4888.5, min: 329, max: 142698
  Words      ->

,topic,n_articles,char_mean,char_median,char_min,char_max,word_mean,word_median,word_min,word_max
0,2_historical_DP_articles_full,733,5850.023192,4355.0,724,122500,964.140518,716.0,112,20142
1,2_historical_climate_articles_full,1559,6455.529827,4602.0,356,124914,1050.849262,741.0,60,22572
2,2_historical_gun_articles_full,1114,8560.536804,4988.5,331,199499,1445.432675,830.5,54,35758
3,DP,699,5161.763948,4395.0,491,51806,854.432046,706.0,85,8219
4,RR,947,6312.807814,5443.0,200,72982,1038.331573,896.0,33,12439
5,climate,1406,6669.317212,4888.5,329,142698,1086.674253,796.0,56,23214
6,gun,907,5421.033076,4616.0,368,60594,893.557883,756.0,55,9924
7,immigration,1684,6722.868171,5225.0,303,79549,1107.434679,852.0,51,12976


In [5]:
# SambaNova client initialization
# Reads API key from sambanova_api_key.txt in this project root.

from pathlib import Path
from sambanova import SambaNova

KEY_FILE = Path("sambanova_api_key.txt")
if not KEY_FILE.exists():
    raise FileNotFoundError(f"Missing {KEY_FILE}. Create it with your SambaNova API key.")

api_key = KEY_FILE.read_text(encoding="utf-8").strip()
if not api_key:
    raise ValueError(f"{KEY_FILE} is empty. Put your SambaNova API key in it.")

client = SambaNova(
    base_url="https://api.sambanova.ai/v1",
    api_key=api_key,
)

print("SambaNova client initialized.")

SambaNova client initialized.


In [6]:
def _json_converter(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return str(obj)

In [ ]:
# DO NOT RUN automatically if you want to control execution manually.
# This cell processes each topic, saves JSON results per topic, and prints anecdote percentages.

llm_model = "gpt-oss-120b"

def parse_llm_output(text: str) -> dict:
    # Expected format: yes/no ; reasoning ; anecdote sentences
    parts = [p.strip() for p in str(text).split(";")]
    label = parts[0].lower() if parts else ""
    has_anecdote = label.startswith("yes")
    reasoning = parts[1] if len(parts) > 1 else ""
    anecdote_sentences = ";".join(parts[2:]).strip() if len(parts) > 2 else ""
    return {
        "raw": text,
        "has_anecdote": has_anecdote,
        "reasoning": reasoning,
        "anecdote_sentences": anecdote_sentences,
    }


for topic, topic_df in articles_df.groupby("topic"):
    topic_df = topic_df.reset_index(drop=True)
    llm_responses_full = []
    topic_results = []

    progress = tqdm(
        enumerate(topic_df["body"]),
        total=len(topic_df),
        desc=f"Topic: {topic}",
        unit="article",
    )

    if topic == "DP":
        continue

    if topic not in ["DP", "RR"]:
        continue

    for i, body in progress:
        response = client.chat.completions.create(
            model=llm_model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant and an expert in storytelling"},
                {"role": "user", "content": (
                    f"An anecdote is a concise story with a point, revealing a factual account of a person"
                    "An anecdote must describe how a person is impacted by a particular issue."
                    "Here is one example: Each summer, Daniel watches the river behind his childhood home rise a little higher. When he was young, flooding was rare — something that happened once in a decade. Now it happens every year. Last spring, he carried his daughter through ankle-deep water to reach their car, trying to make it feel like an adventure. But as he stacked sandbags by the door again, he realized this wasn’t a bad year. It was the new normal."
                    "Here is another example: Louise Yeung relishes the intricacies of policy debates and the magic of rom-coms. She lives in Brooklyn with her cat and two snails. Flooded basements, soaring energy bills, hospital visits resulting from heat, cold and wildfire smoke. "
                    "Read the following article and let me know if it has an anecdote."
                    "Ignore all the quotes in the article."
                    "Reply with only yes or no, followed by your reasoning, followed by the sentences from the article that are anecdotes."
                    "Seperate each section with a ;"
                    f"{body}"
                )}
            ]
        )

        content = response.choices[0].message.content
        llm_responses_full.append(content)

        parsed = parse_llm_output(content)
        topic_results.append({
            "row_index": int(i),
            "ID": topic_df.iloc[i].get("ID"),
            "title": topic_df.iloc[i].get("title"),
            "url": topic_df.iloc[i].get("url"),
            "topic": topic,
            "has_anecdote": parsed["has_anecdote"],
            "reasoning": parsed["reasoning"],
            "anecdote_sentences": parsed["anecdote_sentences"],
            "raw_response": parsed["raw"],
        })

        processed = i + 1
        positives_so_far = sum(1 for r in topic_results if r["has_anecdote"])
        progress.set_postfix(processed=processed, anecdotes=positives_so_far)


    total = len(topic_results)
    positive = sum(1 for r in topic_results if r["has_anecdote"])
    pct = (100.0 * positive / total) if total else 0.0

    output_path = ANECDOTE_RESULTS_DIR / f"{topic}_anecdote_results.json"
    output_path.write_text(
        json.dumps(topic_results, ensure_ascii=False, indent=2, default=_json_converter),
        encoding="utf-8"
    )

    print(f"[{topic}] {positive}/{total} articles with anecdotes ({pct:.2f}%). Saved -> {output_path}")

Topic: DP:   0%|          | 0/699 [00:00<?, ?article/s]

Topic: RR:   0%|          | 0/947 [00:00<?, ?article/s]

[RR] 322/947 articles with anecdotes (34.00%). Saved -> anecdote_outputs/RR_anecdote_results.json


Topic: climate:   0%|          | 0/1406 [00:00<?, ?article/s]

Topic: gun:   0%|          | 0/907 [00:00<?, ?article/s]

Topic: immigration:   0%|          | 0/1684 [00:00<?, ?article/s]

In [7]:
# Incremental anecdote identification for new 2_historical_* files
# Appends only NEW rows into existing per-topic anecdote result files.

import json
from pathlib import Path

NEW_FILE_GLOB = "2_historical_*_articles_full.json"


def topic_from_new_filename(path: Path) -> str:
    # 2_historical_DP_articles_full -> DP
    name = path.stem
    if name.startswith("2_"):
        name = name[2:]
    if name.startswith("historical_") and name.endswith("_articles_full"):
        return name[len("historical_") : -len("_articles_full")]
    return name


def parse_llm_output_incremental(text: str) -> dict:
    parts = [p.strip() for p in str(text).split(";")]
    label = parts[0].lower() if parts else ""
    has_anecdote = label.startswith("yes")
    reasoning = parts[1] if len(parts) > 1 else ""
    anecdote_sentences = ";".join(parts[2:]).strip() if len(parts) > 2 else ""
    return {
        "raw": text,
        "has_anecdote": has_anecdote,
        "reasoning": reasoning,
        "anecdote_sentences": anecdote_sentences,
    }


new_files = sorted(DATA_DIR.glob(NEW_FILE_GLOB))
if not new_files:
    raise FileNotFoundError(f"No files matched: {DATA_DIR / NEW_FILE_GLOB}")

for fp in new_files:
    topic = topic_from_new_filename(fp)
    rows = load_articles_file(fp)
    topic_df = pd.DataFrame(rows).reset_index(drop=True)

    existing_path = ANECDOTE_RESULTS_DIR / f"{topic}_anecdote_results.json"
    existing_rows = []
    if existing_path.exists():
        existing_rows = json.loads(existing_path.read_text(encoding="utf-8"))

    # Deduplicate against already-saved outputs for this topic
    existing_keys = {
        (
            str(r.get("ID", "")),
            str(r.get("url", "")),
            str(r.get("title", "")),
        )
        for r in existing_rows
    }

    to_process_idx = []
    for i, row in topic_df.iterrows():
        key = (
            str(row.get("ID", "")),
            str(row.get("url", "")),
            str(row.get("title", "")),
        )
        if key not in existing_keys:
            to_process_idx.append(i)

    if not to_process_idx:
        print(f"[{topic}] no new rows to process in {fp.name}.")
        continue

    progress = tqdm(to_process_idx, total=len(to_process_idx), desc=f"New file anecdote pass: {topic}", unit="article")
    added_rows = []
    skipped_rows = 0

    for i in progress:
        body = str(topic_df.iloc[i].get("body", "") or "")

        response = client.chat.completions.create(
            model="gpt-oss-120b",
            messages=[
                {"role": "system", "content": "You are a helpful assistant and an expert in storytelling"},
                {"role": "user", "content": (
                    f"An anecdote is a concise story with a point, revealing a factual account of a person"
                    "An anecdote must describe how a person is impacted by a particular issue."
                    "Here is one example: Each summer, Daniel watches the river behind his childhood home rise a little higher. When he was young, flooding was rare — something that happened once in a decade. Now it happens every year. Last spring, he carried his daughter through ankle-deep water to reach their car, trying to make it feel like an adventure. But as he stacked sandbags by the door again, he realized this wasn’t a bad year. It was the new normal."
                    "Here is another example: Louise Yeung relishes the intricacies of policy debates and the magic of rom-coms. She lives in Brooklyn with her cat and two snails. Flooded basements, soaring energy bills, hospital visits resulting from heat, cold and wildfire smoke. "
                    "Read the following article and let me know if it has an anecdote."
                    "Ignore all the quotes in the article."
                    "Reply with only yes or no, followed by your reasoning, followed by the sentences from the article that are anecdotes."
                    "Seperate each section with a ;"
                    f"{body}"
                )}
            ]
        )

        if response is None:
            skipped_rows += 1
            continue

        choices = getattr(response, "choices", None)
        if not choices or choices[0] is None:
            skipped_rows += 1
            continue

        message = getattr(choices[0], "message", None)
        content = getattr(message, "content", None) if message is not None else None
        if content is None:
            skipped_rows += 1
            continue

        parsed = parse_llm_output_incremental(content)

        added_rows.append({
            "row_index": int(i),
            "ID": topic_df.iloc[i].get("ID"),
            "title": topic_df.iloc[i].get("title"),
            "url": topic_df.iloc[i].get("url"),
            "topic": topic,
            "source_file": fp.name,
            "has_anecdote": parsed["has_anecdote"],
            "reasoning": parsed["reasoning"],
            "anecdote_sentences": parsed["anecdote_sentences"],
            "raw_response": parsed["raw"],
        })

    merged = existing_rows + added_rows
    existing_path.write_text(json.dumps(merged, ensure_ascii=False, indent=2, default=_json_converter), encoding="utf-8")

    n_total = len(merged)
    n_added = len(added_rows)
    n_anecdote_added = sum(1 for r in added_rows if r["has_anecdote"])
    print(
        f"[{topic}] appended {n_added} new rows ({n_anecdote_added} anecdotes) from {fp.name}. "
        f"Skipped {skipped_rows} null/empty responses. Total now: {n_total}. Saved -> {existing_path}"
    )

[DP] no new rows to process in 2_historical_DP_articles_full.json.
[climate] no new rows to process in 2_historical_climate_articles_full.json.


New file anecdote pass: gun:   0%|          | 0/1114 [00:00<?, ?article/s]

[gun] appended 1114 new rows (312 anecdotes) from 2_historical_gun_articles_full.json. Skipped 0 null/empty responses. Total now: 2021. Saved -> anecdote_outputs/anecdote_detection_results/gun_anecdote_results.json


In [8]:
# Count anecdotes identified per topic from saved outputs
import json
from pathlib import Path

ANALYSIS_DIR = ANECDOTE_RESULTS_DIR
if not ANALYSIS_DIR.exists():
    raise FileNotFoundError("anecdote_outputs directory not found. Run the anecdote detection cell first.")

anecdote_files = sorted(ANALYSIS_DIR.glob("*_anecdote_results.json"))
if not anecdote_files:
    raise FileNotFoundError("No *_anecdote_results.json files found. Run the anecdote detection cell first.")

counts_summary = []
for fp in anecdote_files:
    topic = fp.name.replace("_anecdote_results.json", "")
    rows = json.loads(fp.read_text(encoding="utf-8"))
    total = len(rows)
    anecdotes = sum(1 for r in rows if bool(r.get("has_anecdote", False)))
    pct = (100.0 * anecdotes / total) if total else 0.0
    counts_summary.append({
        "topic": topic,
        "total_articles": total,
        "anecdote_articles": anecdotes,
        "anecdote_pct": pct,
    })

counts_df = pd.DataFrame(counts_summary).sort_values("topic").reset_index(drop=True)
for _, row in counts_df.iterrows():
    print(f"[{row['topic']}] anecdotes: {int(row['anecdote_articles'])}/{int(row['total_articles'])} ({row['anecdote_pct']:.2f}%)")

counts_df

[DP] anecdotes: 498/1432 (34.78%)
[RR] anecdotes: 322/947 (34.00%)
[climate] anecdotes: 384/2965 (12.95%)
[gun] anecdotes: 500/2021 (24.74%)
[immigration] anecdotes: 367/1684 (21.79%)


,topic,total_articles,anecdote_articles,anecdote_pct
0,DP,1432,498,34.776536
1,RR,947,322,34.002112
2,climate,2965,384,12.951096
3,gun,2021,500,24.740228
4,immigration,1684,367,21.793349


## Extracting Only Anecdote Sentences

The next cell processes only articles that were previously identified as anecdotes and already stance-labeled.
For each such article, it prompts the model to return only the sentences from the article body that are part of the anecdote, filtering out all non-anecdote sentences.
Outputs are saved per topic to `anecdote_outputs/{topic}_anecdote_sentences_only.json`.


In [8]:
# Extract only anecdote sentences from anecdote-identified, stance-labeled articles
import json
from pathlib import Path

stance_files = sorted(ANECDOTE_ONLY_STANCE_DIR.glob("*_anecdote_only_stance.json"))
if not stance_files:
    raise FileNotFoundError("No *_anecdote_only_stance.json files found. Run anecdote-only stance classification first.")


def parse_json_array(raw_text: str) -> list[str]:
    text = str(raw_text).strip()

    # First try: strict JSON array
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except Exception:
        pass

    # Fallback: try extracting first bracketed array
    start = text.find("[")
    end = text.rfind("]")
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end + 1]
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass

    # Final fallback: nothing parseable
    return []


for stance_fp in stance_files:
    topic = stance_fp.name.replace("_anecdote_only_stance.json", "")
    stance_rows = json.loads(stance_fp.read_text(encoding="utf-8"))

    # Process only anecdote-identified rows
    anecdote_rows = [r for r in stance_rows if bool(r.get("has_anecdote", True))]
    topic_df = articles_df[articles_df["topic"] == topic].reset_index(drop=True)

    extracted_rows = []

    progress = tqdm(anecdote_rows, total=len(anecdote_rows), desc=f"Extract anecdote sentences: {topic}", unit="article")
    for row in progress:
        row_index = row.get("row_index")
        stance = str(row.get("stance", "neutral")).strip().lower()

        body = ""
        if isinstance(row_index, int) and 0 <= row_index < len(topic_df):
            body = str(topic_df.iloc[row_index].get("body", "") or "")

        prompt = (
            "You are extracting anecdote-only sentences from a news article.\n\n"
            "Task:\n"
            "Return ONLY the sentences from the article body that are part of an anecdote.\n"
            "An anecdote is a concise factual story about a person and how that person is impacted by the issue.\n"
            "Filter out all non-anecdote content (analysis, background facts, policy discussion, quotes not part of an anecdote, etc.).\n\n"
            "Output format requirements:\n"
            "- Return a valid JSON array of strings.\n"
            "- Each item must be an exact sentence copied from the article body.\n"
            "- Keep original sentence order.\n"
            "- If no anecdote sentences exist, return [] exactly.\n"
            "- Do not include any extra text outside the JSON array.\n\n"
            f"Topic: {topic}\n"
            f"Article body:\n{body}"
        )

        response = client.chat.completions.create(
            model="gpt-oss-120b",
            messages=[
                {"role": "system", "content": "You are a precise text extraction system."},
                {"role": "user", "content": prompt},
            ],
            temperature=0,
        )

        raw = response.choices[0].message.content
        sentences = parse_json_array(raw)
        concatenated = " ".join(sentences)

        extracted_rows.append({
            "topic": topic,
            "stance": stance,
            "row_index": row_index,
            "ID": row.get("ID"),
            "title": row.get("title"),
            "url": row.get("url"),
            "anecdote_sentence_count": len(sentences),
            "anecdote_sentences": sentences,
            "anecdote_text": concatenated,
            "raw_model_output": raw,
        })

    out_path = ANECDOTE_SENTENCES_DIR / f"{topic}_anecdote_sentences_only.json"
    out_path.write_text(json.dumps(extracted_rows, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[{topic}] saved {len(extracted_rows)} rows -> {out_path}")


Extract anecdote sentences: DP:   0%|          | 0/227 [00:00<?, ?article/s]

KeyboardInterrupt: 

In [10]:
# Stats: number of extracted anecdote sentences by topic and stance
import json
from pathlib import Path

sentence_files = sorted(ANECDOTE_SENTENCES_DIR.glob("*_anecdote_sentences_only.json"))
if not sentence_files:
    raise FileNotFoundError("No *_anecdote_sentences_only.json files found. Run extraction cell first.")

all_rows = []
for fp in sentence_files:
    rows = json.loads(fp.read_text(encoding="utf-8"))
    all_rows.extend(rows)

if not all_rows:
    raise ValueError("No extracted anecdote rows found.")

stats_df = pd.DataFrame(all_rows)
stats_df["stance"] = stats_df["stance"].astype(str).str.lower().str.strip()
stats_df["anecdote_sentence_count"] = pd.to_numeric(stats_df["anecdote_sentence_count"], errors="coerce").fillna(0).astype(int)

summary = (
    stats_df.groupby(["topic"], as_index=False)
    .agg(
        n_articles=("row_index", "count"),
        total_sentences=("anecdote_sentence_count", "sum"),
        avg_sentences=("anecdote_sentence_count", "mean"),
        median_sentences=("anecdote_sentence_count", "median"),
        min_sentences=("anecdote_sentence_count", "min"),
        max_sentences=("anecdote_sentence_count", "max"),
    )
    .sort_values(["topic"])
    .reset_index(drop=True)
)

for _, row in summary.iterrows():
    print(
        f"[{row['topic']}] "
        f"articles={int(row['n_articles'])}, total_sentences={int(row['total_sentences'])}, "
        f"avg={row['avg_sentences']:.2f}, median={row['median_sentences']:.2f}, "
        f"min={int(row['min_sentences'])}, max={int(row['max_sentences'])}"
    )

summary

[DP] articles=498, total_sentences=4895, avg=9.83, median=6.00, min=0, max=90
[RR] articles=322, total_sentences=4440, avg=13.79, median=9.00, min=0, max=82
[climate] articles=384, total_sentences=1960, avg=5.10, median=2.00, min=0, max=54
[gun] articles=500, total_sentences=3367, avg=6.73, median=3.00, min=0, max=119
[immigration] articles=367, total_sentences=3119, avg=8.50, median=4.00, min=0, max=129


,topic,n_articles,total_sentences,avg_sentences,median_sentences,min_sentences,max_sentences
0,DP,498,4895,9.829317,6.0,0,90
1,RR,322,4440,13.788820,9.0,0,82
2,climate,384,1960,5.104167,2.0,0,54
3,gun,500,3367,6.734000,3.0,0,119
4,immigration,367,3119,8.498638,4.0,0,129


## Anecdote-Only Stance Classification

The next cell classifies the **stance of the anecdote itself** toward each topic (not the full article stance).
For each anecdote-identified row, it sends only the topic + anecdote text to the model and asks for:
- `stance`: one of `pro`, `against`, `neutral`
- `reasoning`: a short explanation of why that anecdote has that stance

Results are saved per topic as `anecdote_outputs/{topic}_anecdote_only_stance.json`.


In [8]:
# Classify stance of anecdote-only text toward topic (with short reasoning)
import json
from pathlib import Path

anecdote_files = sorted(ANECDOTE_RESULTS_DIR.glob("*_anecdote_results.json"))
if not anecdote_files:
    raise FileNotFoundError("No *_anecdote_results.json files found. Run anecdote detection first.")


def normalize_stance(label: str) -> str:
    x = str(label).strip().lower()
    if x.startswith("pro"):
        return "pro"
    if x.startswith("against") or x.startswith("anti"):
        return "against"
    if x.startswith("neutral") or x.startswith("mixed") or x.startswith("unclear"):
        return "neutral"
    return "neutral"


def parse_stance_response(raw_text: str) -> tuple[str, str]:
    # Expected: stance;reasoning
    text = str(raw_text).strip()
    parts = [p.strip() for p in text.split(";", 1)]
    stance_raw = parts[0] if parts else "neutral"
    reasoning = parts[1] if len(parts) > 1 else ""
    return normalize_stance(stance_raw), reasoning


for fp in anecdote_files:
    topic = fp.name.replace("_anecdote_results.json", "")
    rows = json.loads(fp.read_text(encoding="utf-8"))

    # Only articles previously identified as having an anecdote
    anecdote_rows = [r for r in rows if bool(r.get("has_anecdote", False))]
    topic_out = []

    progress = tqdm(anecdote_rows, total=len(anecdote_rows), desc=f"Anecdote stance: {topic}", unit="anecdote")
    for row in progress:
        anecdote_text = str(row.get("anecdote_sentences", "") or "").strip()
        if not anecdote_text:
            # keep row but mark neutral when anecdote text is missing
            topic_out.append({
                "topic": topic,
                "stance": "neutral",
                "reasoning": "No anecdote text extracted.",
                "anecdote": anecdote_text,
                "row_index": row.get("row_index"),
                "ID": row.get("ID"),
                "title": row.get("title"),
                "url": row.get("url"),
                "raw_model_output": "",
            })
            continue

        prompt = (
            "You are labeling stance of an anecdote toward a policy topic.\n\n"
            "Task:\n"
            "Given a topic and an anecdote excerpt from an article, classify the anecdote's stance toward the topic.\n"
            "This is NOT full-article stance classification. Only use the anecdote text provided.\n\n"
            "Label definitions:\n"
            "- pro: the anecdote supports or argues in favor of the topic position\n"
            "- against: the anecdote opposes or argues against the topic position\n"
            "- neutral: descriptive/mixed/unclear stance\n\n"
            "Output format (strict):\n"
            "stance;reasoning\n"
            "- stance must be exactly one of: pro, against, neutral\n"
            "- reasoning must be one short sentence\n\n"
            f"Topic: {topic}\n"
            f"Anecdote text:\n{anecdote_text}"
        )

        response = client.chat.completions.create(
            model="gpt-oss-120b",
            messages=[
                {"role": "system", "content": "You are a careful political-text classifier."},
                {"role": "user", "content": prompt},
            ],
            temperature=0,
        )

        raw = response.choices[0].message.content
        stance, reasoning = parse_stance_response(raw)

        topic_out.append({
            "topic": topic,
            "stance": stance,
            "reasoning": reasoning,
            "anecdote": anecdote_text,
            "row_index": row.get("row_index"),
            "ID": row.get("ID"),
            "title": row.get("title"),
            "url": row.get("url"),
            "raw_model_output": raw,
        })

    out_path = ANECDOTE_ONLY_STANCE_DIR / f"{topic}_anecdote_only_stance.json"
    out_path.write_text(json.dumps(topic_out, ensure_ascii=False, indent=2), encoding="utf-8")

    n = len(topic_out)
    pro_n = sum(1 for r in topic_out if r["stance"] == "pro")
    against_n = sum(1 for r in topic_out if r["stance"] == "against")
    neutral_n = sum(1 for r in topic_out if r["stance"] == "neutral")

    print(f"[{topic}] saved {n} -> {out_path} (pro={pro_n}, against={against_n}, neutral={neutral_n})")

Anecdote stance: DP:   0%|          | 0/498 [00:00<?, ?anecdote/s]

KeyboardInterrupt: 

In [7]:
# Stats from anecdote-only stance files: per-topic counts of pro/against/neutral
import json
from pathlib import Path

stance_files = sorted(ANECDOTE_ONLY_STANCE_DIR.glob("*_anecdote_only_stance.json"))
if not stance_files:
    raise FileNotFoundError("No *_anecdote_only_stance.json files found. Run anecdote-only stance cell first.")

summary_rows = []
for fp in stance_files:
    topic = fp.name.replace("_anecdote_only_stance.json", "")
    rows = json.loads(fp.read_text(encoding="utf-8"))

    pro_n = sum(1 for r in rows if str(r.get("stance", "")).strip().lower() == "pro")
    against_n = sum(1 for r in rows if str(r.get("stance", "")).strip().lower() == "against")
    neutral_n = sum(1 for r in rows if str(r.get("stance", "")).strip().lower() == "neutral")

    summary_rows.append({
        "topic": topic,
        "total_anecdotes": len(rows),
        "pro": pro_n,
        "against": against_n,
        "neutral": neutral_n,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("topic").reset_index(drop=True)

for _, row in summary_df.iterrows():
    print(
        f"[{row['topic']}] total={int(row['total_anecdotes'])} | "
        f"pro={int(row['pro'])}, against={int(row['against'])}, neutral={int(row['neutral'])}"
    )

summary_df

[DP] total=498 | pro=22, against=146, neutral=330
[RR] total=322 | pro=86, against=62, neutral=174
[climate] total=384 | pro=68, against=21, neutral=295
[gun] total=500 | pro=84, against=69, neutral=347
[immigration] total=367 | pro=65, against=73, neutral=229


,topic,total_anecdotes,pro,against,neutral
0,DP,498,22,146,330
1,RR,322,86,62,174
2,climate,384,68,21,295
3,gun,500,84,69,347
4,immigration,367,65,73,229
